# DenStream — Density-Based Clustering for Data Streams

> Cao, F. et al. (2006) *"DenStream: A Clustering Algorithm over an Evolving Data Stream"*

---

## Motivation

Traditional DBSCAN requires the **full dataset in memory** — impossible for infinite streams.  
DenStream solves this by maintaining compact **micro-clusters** with a **fading function** that automatically forgets old data.

---

## Key concepts

### Fading function

Each data point's contribution decays exponentially with time:

$$f(t) = 2^{-\lambda t}, \quad \lambda > 0$$

### Micro-cluster (MC)

A micro-cluster for a set of points $\{p_1, p_2, \ldots, p_n\}$ arriving at times $\{t_1, \ldots, t_n\}$ at current time $t_c$ is defined by its **sufficient statistics**:

| Statistic | Formula |
|---|---|
| Weight | $w = \sum_{i=1}^n f(t_c - t_i)$ |
| Weighted centroid | $\bar{x} = \frac{1}{w}\sum_{i=1}^n f(t_c - t_i) \cdot p_i$ |
| Radius | $r = \frac{1}{w}\sum_{i=1}^n f(t_c - t_i) \cdot \|p_i - \bar{x}\|$ |

### Types of micro-clusters

| Type | Condition | Role |
|---|---|---|
| **Core-micro-cluster** | $w \geq \mu$ (dense enough) | Seed for final clusters |
| **Potential-micro-cluster (p-MC)** | $w \geq \beta \mu$ | Candidate; tracked closely |
| **Outlier-micro-cluster (o-MC)** | $w < \beta \mu$ | May be noise; periodically pruned |

---

## Algorithm: Two-phase

1. **Online phase** — Process each new point:
   - Try to merge with nearest p-MC (within radius $\varepsilon$)
   - If no fit: try nearest o-MC
   - If still no fit: create a new o-MC
   - Periodically prune o-MCs that cannot become dense

2. **Offline phase** — On user request:
   - Apply DBSCAN to the centroids of all p-MCs
   - Weights serve as density estimates

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Imports OK')

---
## 1  Fading Function Visualisation

In [ ]:
t = np.linspace(0, 20, 300)

plt.figure(figsize=(9, 4))
for lam in [0.1, 0.25, 0.5, 1.0]:
    plt.plot(t, 2**(-lam * t), label=f'λ={lam}')

plt.xlabel('Time elapsed since arrival')
plt.ylabel('f(t) = 2^{-λt}')
plt.title('Fading Function — Effect of λ')
plt.legend()
plt.tight_layout()
plt.show()

---
## 2  Micro-Cluster Implementation

In [ ]:
class MicroCluster:
    """DenStream micro-cluster with fading sufficient statistics."""

    def __init__(self, point: np.ndarray, timestamp: float, lam: float = 0.25):
        self.lam = lam
        self.cf1 = point.copy().astype(float)   # weighted sum of points
        self.cf2 = (point ** 2).astype(float)   # weighted sum of squares
        self.w = 1.0                             # total weight
        self.last_update = timestamp

    def _fade(self, t_now: float) -> None:
        """Apply fading for time elapsed since last update."""
        dt = t_now - self.last_update
        factor = 2.0 ** (-self.lam * dt)
        self.cf1 *= factor
        self.cf2 *= factor
        self.w   *= factor
        self.last_update = t_now

    def insert(self, point: np.ndarray, t_now: float) -> None:
        """Increment operator — add new point to micro-cluster."""
        self._fade(t_now)
        self.cf1 += point
        self.cf2 += point ** 2
        self.w   += 1.0

    @property
    def centroid(self) -> np.ndarray:
        return self.cf1 / self.w

    @property
    def radius(self) -> float:
        """Root of weighted variance across all dimensions."""
        var = np.maximum(self.cf2 / self.w - (self.cf1 / self.w) ** 2, 0)
        return float(np.sqrt(np.mean(var)))

    def weight_at(self, t_now: float) -> float:
        dt = t_now - self.last_update
        return self.w * 2.0 ** (-self.lam * dt)


# Quick sanity check
mc = MicroCluster(np.array([1.0, 2.0]), timestamp=0, lam=0.25)
mc.insert(np.array([1.5, 2.5]), t_now=1)
mc.insert(np.array([0.8, 1.8]), t_now=2)
print(f'Centroid : {mc.centroid}')
print(f'Radius   : {mc.radius:.4f}')
print(f'Weight   : {mc.w:.4f}')

---
## 3  DenStream — Online Phase

In [ ]:
class DenStream:
    """
    Simplified DenStream implementation.
    Maintains potential (p-MCs) and outlier (o-MCs) micro-clusters.
    """

    def __init__(self, lam: float = 0.25, epsilon: float = 0.5,
                 mu: float = 2.0, beta: float = 0.5):
        self.lam = lam
        self.epsilon = epsilon   # neighbourhood radius
        self.mu = mu             # minimum weight for core-MC
        self.beta = beta         # multiplier: potential if w >= beta*mu
        self.p_mcs: list[MicroCluster] = []
        self.o_mcs: list[MicroCluster] = []
        self._t = 0

    def _nearest(self, point, clusters):
        if not clusters:
            return None, np.inf
        dists = [np.linalg.norm(mc.centroid - point) for mc in clusters]
        idx = int(np.argmin(dists))
        return clusters[idx], dists[idx]

    def update(self, point: np.ndarray) -> None:
        self._t += 1
        t = self._t

        # Try to merge with nearest p-MC
        mc, d = self._nearest(point, self.p_mcs)
        if mc is not None and d <= self.epsilon:
            mc.insert(point, t)
            return

        # Try nearest o-MC
        mc, d = self._nearest(point, self.o_mcs)
        if mc is not None and d <= self.epsilon:
            mc.insert(point, t)
            # Promote to p-MC if dense enough
            if mc.weight_at(t) >= self.beta * self.mu:
                self.p_mcs.append(mc)
                self.o_mcs.remove(mc)
            return

        # Create new o-MC
        self.o_mcs.append(MicroCluster(point, t, self.lam))

    def prune_outliers(self) -> None:
        """Remove o-MCs that cannot reach beta*mu weight before fading away."""
        t = self._t
        threshold = self.beta * self.mu
        self.o_mcs = [mc for mc in self.o_mcs
                      if mc.weight_at(t) >= threshold * 0.1]

    def get_clusters(self, dbscan_eps: float = 1.5, min_samples: int = 1):
        """Offline phase: run DBSCAN on p-MC centroids."""
        if not self.p_mcs:
            return np.array([]), np.array([])
        centroids = np.array([mc.centroid for mc in self.p_mcs])
        weights   = np.array([mc.w for mc in self.p_mcs])
        labels = DBSCAN(eps=dbscan_eps, min_samples=min_samples).fit_predict(centroids)
        return centroids, labels


print('DenStream class ready.')

---
## 4  Stream Simulation & Clustering

In [ ]:
# Generate a 2-D stream with 3 evolving clusters
def make_stream(n=800, seed=42):
    rng = np.random.default_rng(seed)
    centers = [(-3, -3), (0, 3), (3, -1)]
    X, labels = [], []
    for i in range(n):
        c_idx = i % 3
        cx, cy = centers[c_idx]
        X.append([cx + rng.normal(0, 0.6), cy + rng.normal(0, 0.6)])
        labels.append(c_idx)
    return np.array(X), np.array(labels)

X_stream, true_labels = make_stream()

ds = DenStream(lam=0.25, epsilon=1.2, mu=3.0, beta=0.5)

for i, point in enumerate(X_stream):
    ds.update(point)
    if (i + 1) % 100 == 0:
        ds.prune_outliers()

print(f'p-MCs: {len(ds.p_mcs)}  |  o-MCs: {len(ds.o_mcs)}')

centroids, cluster_labels = ds.get_clusters(dbscan_eps=2.0)
print(f'Final clusters found: {len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Raw stream (last 200 points)
axes[0].scatter(X_stream[-200:, 0], X_stream[-200:, 1],
                c=true_labels[-200:], cmap='tab10', alpha=0.5, s=15)
axes[0].set_title('Raw Stream (last 200 points, true labels)')

# DenStream micro-clusters
pmc_centroids = np.array([mc.centroid for mc in ds.p_mcs])
pmc_weights   = np.array([mc.w for mc in ds.p_mcs])
axes[1].scatter(X_stream[:, 0], X_stream[:, 1], alpha=0.1, s=5, color='grey')
sc = axes[1].scatter(pmc_centroids[:, 0], pmc_centroids[:, 1],
                     c=cluster_labels, cmap='tab10', s=pmc_weights * 20,
                     edgecolors='k', zorder=5)
axes[1].set_title('DenStream p-MC Centroids (size ∝ weight)')

plt.tight_layout()
plt.show()

---
## 5  Effect of λ on Cluster Recency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, lam in zip(axes, [0.05, 0.5]):
    ds_tmp = DenStream(lam=lam, epsilon=1.2, mu=3.0, beta=0.5)
    for i, p in enumerate(X_stream):
        ds_tmp.update(p)
        if (i+1) % 100 == 0:
            ds_tmp.prune_outliers()

    if ds_tmp.p_mcs:
        c = np.array([mc.centroid for mc in ds_tmp.p_mcs])
        w = np.array([mc.w for mc in ds_tmp.p_mcs])
        ax.scatter(X_stream[:, 0], X_stream[:, 1], alpha=0.1, s=5, color='grey')
        ax.scatter(c[:, 0], c[:, 1], s=w*15, edgecolors='k', zorder=5)
    ax.set_title(f'λ = {lam}  ({len(ds_tmp.p_mcs)} p-MCs)')

plt.suptitle('Higher λ → faster forgetting → fewer, more recent micro-clusters')
plt.tight_layout()
plt.show()

---
## Summary

| Component | Role |
|---|---|
| Fading function $2^{-\lambda t}$ | Automatically forgets old data |
| Sufficient statistics (CF1, CF2, w) | O(1) memory per micro-cluster |
| p-MC (potential) | Dense; seed for final clusters |
| o-MC (outlier) | Sparse; promoted or pruned |
| Offline DBSCAN on p-MCs | Final cluster assignment on demand |

**See also:** Topic 36 (DBSCAN), Topic 117 (stream adaptive learning)